# Bottle Cap Detection Pipeline

This notebook implements a complete pipeline for bottle cap detection:
1. **Video Frame Extraction** - Extract JPEG frames from video every 0.5 seconds
2. **Auto-Labeling with Autodistill** - Use GroundingDINO to automatically label bottle caps
3. **Dataset Preparation** - Prepare data in YOLO format with train/validation split
4. **YOLOv11 Training** - Train a custom object detection model
5. **Model Evaluation** - Test and validate the trained model

## Prerequisites
- Video file named `caps.mov` in the current directory
- Python environment with required packages (installed in first cell)

## Stage 0: Package Installation

Install all required packages for the complete pipeline.

In [ ]:
# Install required packages
%pip install opencv-python numpy matplotlib pathlib ipython
%pip install autodistill autodistill-grounding-dino
%pip install ultralytics roboflow supervision
%pip install pillow pyyaml sklearn

In [ ]:
# Import all required libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display
import shutil
import yaml
from PIL import Image as PILImage
import supervision as sv

# Autodistill imports
from autodistill.detection import CaptionOntology
from autodistill_grounding_dino import GroundingDINO

# Ultralytics YOLO
from ultralytics import YOLO

print("All libraries imported successfully!")

## Stage 1: Video Frame Extraction

Extract JPEG frames from the video file every 0.5 seconds for training data.

### 1.1: Check Source Video FIle

In [ ]:
# Configuration for frame extraction
VIDEO_PATH = "caps.mov"  # Path to the input video
EXTRACTED_FRAMES_DIR = "frames"  # Directory to save extracted frames
EXTRACTION_INTERVAL = 0.5  # Extract frame every 0.5 seconds
IMAGE_FORMAT = "jpg"  # Output image format

# Create output directory if it doesn't exist
Path(EXTRACTED_FRAMES_DIR).mkdir(exist_ok=True)

# Check if video file exists
if not os.path.exists(VIDEO_PATH):
    print(f"Error: Video file '{VIDEO_PATH}' not found!")
else:
    # Get video file size
    file_size = os.path.getsize(VIDEO_PATH) / (1024 * 1024)  # MB
    print(f"Video file size: {file_size:.2f} MB")

# Open video file and get properties
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    print("Error: Could not open video file")
else:
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f"FPS: {fps}")
    print(f"Total frames: {frame_count}")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Resolution: {width}x{height}")

    cap.release()

### 1.2: Extract Frames from Video

In [ ]:
def extract_frames(video_path, output_dir, interval_seconds=0.5):
    """
    Extract frames from video at specified intervals.
    
    Args:
        video_path (str): Path to input video file
        output_dir (str): Directory to save extracted frames
        interval_seconds (float): Time interval between extracted frames
    
    Returns:
        list: List of extracted frame filenames
    """
    # make sure folder exists
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"Error: Could not open video file {video_path}")
        return []
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_interval = int(fps * interval_seconds)
    extracted_files = []
    frame_number = 0
    extracted_count = 0
    
    print(f"Starting extraction... (interval: {frame_interval} frames at {fps} FPS)")
    print(f"Expected to extract approximately {total_frames // frame_interval} frames")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_number % frame_interval == 0:
            # Calculate time for this frame
            time_seconds = frame_number / fps
            filename = f"frame_{extracted_count:04d}.jpg"
            filepath = os.path.join(output_dir, filename)
            
            if cv2.imwrite(filepath, frame):
                extracted_files.append(filename)
                extracted_count += 1
                
                # Progress indicator every 20 frames
                if extracted_count % 20 == 0:
                    print(f"  Extracted {extracted_count} frames...")
        
        frame_number += 1
    
    cap.release()
    print(f"\nCompleted! {extracted_count} frames saved in: {output_dir}")
    
    return extracted_files

In [ ]:
# Extract frames from the video
extracted_files = extract_frames(VIDEO_PATH, EXTRACTED_FRAMES_DIR, EXTRACTION_INTERVAL)

if extracted_files:
    print(f"\nFirst few extracted files:")
    for i, filename in enumerate(extracted_files[:5]):
        print(f"  {i+1}. {filename}")
    
    if len(extracted_files) > 5:
        print(f"  ... and {len(extracted_files) - 5} more files")
else:
    print("Frame extraction failed!")

### 1.3: Show Extracted Frames

In [ ]:
# Display a sample of extracted frames
if extracted_files:
    print("Sample of extracted frames:")
    
    # Show first 6 frames in a grid
    sample_count = min(6, len(extracted_files))
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i in range(sample_count):
        if i < len(extracted_files):
            filepath = os.path.join(EXTRACTED_FRAMES_DIR, extracted_files[i])
            
            # Read image using OpenCV and convert BGR to RGB for matplotlib
            img = cv2.imread(filepath)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            axes[i].imshow(img_rgb)
            axes[i].set_title(extracted_files[i], fontsize=10)
            axes[i].axis('off')
        else:
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("No frames were extracted.")

## Stage 2: Auto-Labeling with Autodistill

Use GroundingDINO foundation model to automatically detect and label bottle caps in the extracted frames.

### 2.1: Basic Configuration

Set up the basic dataset directory. Other configuration constants are defined closer to where they're used throughout the notebook.

In [ ]:
# Basic dataset configuration
DATASET_DIR = "frames_labeled"

print(f"Dataset directory: {DATASET_DIR}")

# Define directory structure for GroundingDINO output
# Autodistill structure: images directly in DATASET_DIR, labels in DATASET_DIR/labels
AUTODISTILL_LABELS_DIR = os.path.join(DATASET_DIR, "labels")
AUTODISTILL_IMAGES_DIR = DATASET_DIR  # Images are directly in dataset root

# YOLO training structure after splitting
TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "valid")
TRAIN_IMAGES_DIR = os.path.join(TRAIN_DIR, "images")
TRAIN_LABELS_DIR = os.path.join(TRAIN_DIR, "labels")
VAL_IMAGES_DIR = os.path.join(VAL_DIR, "images")
VAL_LABELS_DIR = os.path.join(VAL_DIR, "labels")

print(f"Autodistill images: {AUTODISTILL_IMAGES_DIR}")
print(f"Autodistill labels: {AUTODISTILL_LABELS_DIR}")

# Create necessary directories for YOLO training structure
directories = [
    DATASET_DIR,
    AUTODISTILL_LABELS_DIR,
    TRAIN_DIR,
    VAL_DIR,
    TRAIN_IMAGES_DIR,
    TRAIN_LABELS_DIR,
    VAL_IMAGES_DIR,
    VAL_LABELS_DIR
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)
    print(f"Created directory: {directory}")

print("\nDirectory structure created successfully!")
print("\nExpected structure after autodistill:")
print(f"  {AUTODISTILL_IMAGES_DIR}/*.jpg (images)")
print(f"  {AUTODISTILL_LABELS_DIR}/*.txt (labels)")
print("\nStructure after train/val split:")
print(f"  {TRAIN_IMAGES_DIR}/*.jpg")
print(f"  {TRAIN_LABELS_DIR}/*.txt")
print(f"  {VAL_IMAGES_DIR}/*.jpg")
print(f"  {VAL_LABELS_DIR}/*.txt")

### 2.2: Configure GroundingDINO

In [ ]:
# Class definitions
CLASS_NAMES = ["bottle_cap"]
NUM_CLASSES = len(CLASS_NAMES)

# Set up Autodistill with GroundingDINO
# Define ontology - what we want to detect
ontology = CaptionOntology({
    "round plastic piece": "bottle_cap",
})

# Initialize the base model
base_model = GroundingDINO(ontology=ontology)

print(f"Classes: {CLASS_NAMES}")
print("Autodistill GroundingDINO model initialized!")
print(f"Ontology: {ontology.prompts()}")

### 2.3: Test GroundingDINO

In [ ]:
# Test autodistill on a sample image first
if extracted_files:
    # Load a sample image for testing
    sample_image_path = os.path.join(EXTRACTED_FRAMES_DIR, extracted_files[5])
    print(f"Testing autodistill on: {sample_image_path}")
    
    # Run detection
    detections = base_model.predict(sample_image_path)
    
    # Load image for visualization
    image = cv2.imread(sample_image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Annotate image with detections
    annotated_image = sv.BoxAnnotator().annotate(
        scene=image_rgb,
        detections=detections
    )
    
    # Display results
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_image)
    plt.title(f"Autodistill Detection Test - Found {len(detections)} bottle caps")
    plt.axis('off')
    plt.show()
    
    print(f"Detection results:")
    print(f"  Number of detections: {len(detections)}")
    if len(detections) > 0:
        print(f"  Confidence scores: {detections.confidence}")
        print(f"  Bounding boxes: {detections.xyxy}")
else:
    print("No frames available for testing.")

### 2.4: Run Auto-Labeling

In [ ]:
# Auto-label all extracted frames
if extracted_files:
    print(f"Starting auto-labeling for {len(extracted_files)} frames...")

    # Run autodistill on the entire dataset
    base_model.label(input_folder=EXTRACTED_FRAMES_DIR, output_folder=DATASET_DIR)

    print("Auto-labeling completed!")

    # Check generated labels in the autodistill structure
    if os.path.exists(AUTODISTILL_LABELS_DIR):
        label_files = [
            f for f in os.listdir(AUTODISTILL_LABELS_DIR) if f.endswith(".txt")
        ]
        print(f"Generated {len(label_files)} label files in {AUTODISTILL_LABELS_DIR}")

        # Check for images in dataset root
        image_files = [
            f
            for f in os.listdir(AUTODISTILL_IMAGES_DIR)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ]
        print(f"Found {len(image_files)} images in {AUTODISTILL_IMAGES_DIR}")
    else:
        print(f"Warning: Expected labels directory {AUTODISTILL_LABELS_DIR} not found!")
else:
    print("No frames available for labeling.")

# Dataset Analysis - Analyze the auto-labeled dataset
print("\n" + "=" * 50)
print("DATASET ANALYSIS")
print("=" * 50)


### 2.5: Organize GroundingDINO Output

Let's create a clean `frames_labeled` directory with the raw GroundingDINO output, then create a proper train/validation split.

In [ ]:
# Create frames_labeled directory with all labeled data from GroundingDINO
FRAMES_LABELED_DIR = "frames_labeled"
FRAMES_LABELED_IMAGES_DIR = os.path.join(FRAMES_LABELED_DIR, "images")
FRAMES_LABELED_LABELS_DIR = os.path.join(FRAMES_LABELED_DIR, "labels")

# Create directories
Path(FRAMES_LABELED_DIR).mkdir(exist_ok=True)
Path(FRAMES_LABELED_IMAGES_DIR).mkdir(exist_ok=True)
Path(FRAMES_LABELED_LABELS_DIR).mkdir(exist_ok=True)

# Collect all labeled data from the current structure
all_images = []
all_labels = []

# Check both train and valid directories from GroundingDINO output
for split_dir in ['train', 'valid']:
    images_path = os.path.join(DATASET_DIR, split_dir, 'images')
    labels_path = os.path.join(DATASET_DIR, split_dir, 'labels')
    
    if os.path.exists(images_path):
        for img_file in os.listdir(images_path):
            if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_images.append(os.path.join(images_path, img_file))
    
    if os.path.exists(labels_path):
        for label_file in os.listdir(labels_path):
            if label_file.endswith('.txt'):
                all_labels.append(os.path.join(labels_path, label_file))

print(f"Found {len(all_images)} images and {len(all_labels)} labels from GroundingDINO")

# Copy all images and labels to frames_labeled directory
for img_path in all_images:
    img_name = os.path.basename(img_path)
    dst_path = os.path.join(FRAMES_LABELED_IMAGES_DIR, img_name)
    shutil.copy2(img_path, dst_path)

for label_path in all_labels:
    label_name = os.path.basename(label_path)
    dst_path = os.path.join(FRAMES_LABELED_LABELS_DIR, label_name)
    shutil.copy2(label_path, dst_path)

print(f"Copied all labeled data to {FRAMES_LABELED_DIR}")
print(f"  Images: {FRAMES_LABELED_IMAGES_DIR}")
print(f"  Labels: {FRAMES_LABELED_LABELS_DIR}")

## Stage 3: Dataset Preparation

Now let's create a proper train/validation split with shuffled data in `frames_labeled_split`.

In [ ]:
# Create frames_labeled_split directory with proper train/validation split
FRAMES_LABELED_SPLIT_DIR = "frames_labeled_split"
SPLIT_TRAIN_DIR = os.path.join(FRAMES_LABELED_SPLIT_DIR, "train")
SPLIT_VAL_DIR = os.path.join(FRAMES_LABELED_SPLIT_DIR, "valid")
SPLIT_TRAIN_IMAGES_DIR = os.path.join(SPLIT_TRAIN_DIR, "images")
SPLIT_TRAIN_LABELS_DIR = os.path.join(SPLIT_TRAIN_DIR, "labels")
SPLIT_VAL_IMAGES_DIR = os.path.join(SPLIT_VAL_DIR, "images")
SPLIT_VAL_LABELS_DIR = os.path.join(SPLIT_VAL_DIR, "labels")

# Create directory structure
directories = [
    FRAMES_LABELED_SPLIT_DIR,
    SPLIT_TRAIN_DIR,
    SPLIT_VAL_DIR,
    SPLIT_TRAIN_IMAGES_DIR,
    SPLIT_TRAIN_LABELS_DIR,
    SPLIT_VAL_IMAGES_DIR,
    SPLIT_VAL_LABELS_DIR
]

for directory in directories:
    Path(directory).mkdir(parents=True, exist_ok=True)

# Get all image files from frames_labeled
image_files = [f for f in os.listdir(FRAMES_LABELED_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Check that we have corresponding labels for all images
valid_pairs = []
for img_file in image_files:
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = os.path.join(FRAMES_LABELED_LABELS_DIR, label_file)
    if os.path.exists(label_path):
        valid_pairs.append((img_file, label_file))

print(f"Found {len(valid_pairs)} valid image-label pairs")

# Shuffle the data for random split
np.random.seed(42)  # For reproducible results
np.random.shuffle(valid_pairs)

# Split data (80% train, 20% validation)
train_split = 0.8
split_index = int(len(valid_pairs) * train_split)
train_pairs = valid_pairs[:split_index]
val_pairs = valid_pairs[split_index:]

print(f"Train samples: {len(train_pairs)}")
print(f"Validation samples: {len(val_pairs)}")

# Copy training data
for img_file, label_file in train_pairs:
    # Copy image
    src_img = os.path.join(FRAMES_LABELED_IMAGES_DIR, img_file)
    dst_img = os.path.join(SPLIT_TRAIN_IMAGES_DIR, img_file)
    shutil.copy2(src_img, dst_img)
    
    # Copy label
    src_label = os.path.join(FRAMES_LABELED_LABELS_DIR, label_file)
    dst_label = os.path.join(SPLIT_TRAIN_LABELS_DIR, label_file)
    shutil.copy2(src_label, dst_label)

# Copy validation data
for img_file, label_file in val_pairs:
    # Copy image
    src_img = os.path.join(FRAMES_LABELED_IMAGES_DIR, img_file)
    dst_img = os.path.join(SPLIT_VAL_IMAGES_DIR, img_file)
    shutil.copy2(src_img, dst_img)
    
    # Copy label
    src_label = os.path.join(FRAMES_LABELED_LABELS_DIR, label_file)
    dst_label = os.path.join(SPLIT_VAL_LABELS_DIR, label_file)
    shutil.copy2(src_label, dst_label)

print(f"\nDataset split completed!")
print(f"Train data: {SPLIT_TRAIN_IMAGES_DIR}")
print(f"Validation data: {SPLIT_VAL_IMAGES_DIR}")

# Store counts for later use
train_count = len(train_pairs)
val_count = len(val_pairs)

In [ ]:
# Update the training directory variables to use the new split
TRAIN_IMAGES_DIR = SPLIT_TRAIN_IMAGES_DIR
TRAIN_LABELS_DIR = SPLIT_TRAIN_LABELS_DIR
VAL_IMAGES_DIR = SPLIT_VAL_IMAGES_DIR
VAL_LABELS_DIR = SPLIT_VAL_LABELS_DIR

print(f"Updated training directories:")
print(f"  Train images: {TRAIN_IMAGES_DIR}")
print(f"  Train labels: {TRAIN_LABELS_DIR}")
print(f"  Val images: {VAL_IMAGES_DIR}")
print(f"  Val labels: {VAL_LABELS_DIR}")

# Analyze the dataset
dataset_stats = {
    'total_images': train_count + val_count,
    'train_images': train_count,
    'val_images': val_count,
    'total_objects': 0,
    'objects_per_image': []
}

# Count objects in all label files
all_label_files = []
all_label_files.extend([os.path.join(TRAIN_LABELS_DIR, f) for f in os.listdir(TRAIN_LABELS_DIR) if f.endswith('.txt')])
all_label_files.extend([os.path.join(VAL_LABELS_DIR, f) for f in os.listdir(VAL_LABELS_DIR) if f.endswith('.txt')])

for label_file in all_label_files:
    with open(label_file, 'r') as f:
        lines = f.readlines()
        objects_count = len([line for line in lines if line.strip()])
        dataset_stats['total_objects'] += objects_count
        dataset_stats['objects_per_image'].append(objects_count)

print(f"\nDataset Statistics:")
print(f"  Total images: {dataset_stats['total_images']}")
print(f"  Training images: {dataset_stats['train_images']}")
print(f"  Validation images: {dataset_stats['val_images']}")
print(f"  Total bottle caps detected: {dataset_stats['total_objects']}")
if dataset_stats['objects_per_image']:
    print(f"  Average caps per image: {np.mean(dataset_stats['objects_per_image']):.1f}")
    print(f"  Min caps per image: {min(dataset_stats['objects_per_image'])}")
    print(f"  Max caps per image: {max(dataset_stats['objects_per_image'])}")

## Stage 4: YOLOv11 Model Training


In [ ]:
# Create YOLO dataset configuration file
def create_yolo_config(config_path, dataset_dir, class_names):
    """
    Create YOLO dataset configuration YAML file.
    """
    config = {
        'path': os.path.abspath(dataset_dir),
        'train': 'train/images',
        'val': 'valid/images',
        'nc': len(class_names),
        'names': {i: name for i, name in enumerate(class_names)}
    }
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)
    
    print(f"YOLO config saved to: {config_path}")
    print(f"Dataset path: {config['path']}")
    print(f"Train path: {config['train']}")
    print(f"Val path: {config['val']}")
    print(f"Classes: {config['names']}")
    
    return config

# Define YOLO config file path and create the configuration
YOLO_CONFIG_FILE = os.path.join(FRAMES_LABELED_SPLIT_DIR, "dataset.yaml")

# Create the configuration
yolo_config = create_yolo_config(YOLO_CONFIG_FILE, FRAMES_LABELED_SPLIT_DIR, CLASS_NAMES)

# Display the configuration
print("\nYOLO Configuration:")
with open(YOLO_CONFIG_FILE, 'r') as f:
    print(f.read())

In [ ]:
# Training parameters
EPOCHS = 100
BATCH_SIZE = 16
IMAGE_SIZE = 640
MODEL_OUTPUT_DIR = "yolo_models"

print(f"Training epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMAGE_SIZE}")

# Initialize and train YOLOv11 model
print("Initializing YOLOv11 model...")

# Load a pre-trained YOLOv11 model
model = YOLO('yolo11n.pt')  # nano version for faster training

print(f"Model loaded: {model.model_name}")
print(f"Model summary:")
model.info()

In [ ]:
# Train the model
if 'train_count' in locals() and 'val_count' in locals() and train_count > 0 and val_count > 0:
    print(f"Starting training with {EPOCHS} epochs...")
    
    # Train the model
    results = model.train(
        data=YOLO_CONFIG_FILE,
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        save=True,
        project=MODEL_OUTPUT_DIR,
        name='bottle_cap_detection',
        patience=20,  # Early stopping patience
        save_period=10,  # Save checkpoint every 10 epochs
        plots=True  # Generate training plots
    )
    
    print("Training completed!")
    print(f"Model saved in: {MODEL_OUTPUT_DIR}")
else:
    print("Insufficient data for training. Need both training and validation samples.")
    results = None

In [ ]:
# Load the trained model and run validation
if 'results' in locals() and results is not None:
    # Load the best trained model
    best_model_path = os.path.join(MODEL_OUTPUT_DIR, 'bottle_cap_detection', 'weights', 'best.pt')
    
    if os.path.exists(best_model_path):
        trained_model = YOLO(best_model_path)
        print(f"Loaded trained model from: {best_model_path}")
        
        # Run validation
        val_results = trained_model.val(data=YOLO_CONFIG_FILE)
        
        print("\nValidation Results:")
        print(f"mAP50: {val_results.box.map50:.4f}")
        print(f"mAP50-95: {val_results.box.map:.4f}")
        print(f"Precision: {val_results.box.mp:.4f}")
        print(f"Recall: {val_results.box.mr:.4f}")
    else:
        print(f"Trained model not found at: {best_model_path}")
        trained_model = None
else:
    print("No trained model available for validation.")
    trained_model = None